# Exploration de l'API SNCF

Objectifs et ordre détaillés dans `notebooks/README.md`.
Le point le plus critique est le point 4 : sans identifiant de circulation persistant d'une gare à l'autre, l'hypothèse de propagation ne tient pas.

In [60]:
%reload_ext autoreload
%autoreload 2
from dotenv import load_dotenv

load_dotenv()

from collector.sncf_client import SncfClient
from collector.stations import STATIONS

client = SncfClient()

## 1. Résoudre les codes UIC réels des cinq gares via l'endpoint `places`

In [63]:
for station in STATIONS:
    place = client.search_places(station.label)
    try :
        stop_area_codes = place['places'][0]['stop_area']['codes']
        uic_code = [p for p in stop_area_codes if p["type"] == "uic"]
        uic = uic_code[0]['value'] if uic_code else None

        if  uic != station.uic:
            print(f"{station.label}: {station.uic} devrait plutôt etre cet uic : {uic}")
        else:
            print(f"Code uic correct pour {station.label}: {station.uic}")
    except (KeyError, IndexError):
        print(f"Mauvais parsing pour ce json:\n{place}")

Code uic correct pour Bordeaux Saint-Jean: 87581009
Code uic correct pour Toulouse Matabiau: 87611004
Code uic correct pour Montpellier Saint-Roch: 87773002
Code uic correct pour Marseille Saint-Charles: 87751008
Code uic correct pour Antibes: 87757674


## 2. Appeler `departures` sur une gare et inspecter la structure complète

## 3. Vérifier que `data_freshness=realtime` renvoie un horaire différent du théorique quand un train est en retard

## 4. Identifier le champ portant l'identifiant de circulation persistant d'une gare à l'autre

Point critique : condition nécessaire à tout le projet.

## 5. Mesurer le quota réel : combien d'appels avant un HTTP 429

## 6. Vérifier si le quai (`stop_point`) est renseigné en temps réel